In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
import os


MODEL_NAME = "google/gemma-4-E4B-it"

c:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_NAME).to("cuda")

NameError: name 'AutoProcessor' is not defined

In [ ]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

# Example

In [ ]:
messages = [
    {"role": "system", "content": "You are helpful assistant."},
    {"role": "user", "content": "Write a short joke about saving RAM."}
]

messages = [messages, messages]

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

In [ ]:
inputs

{'input_ids': tensor([[     2,    105,   9731,    107,   3048,    659,  11045,  16326, 236761,
            106,    107,    105,   2364,    107,   6974,    496,   2822,  31481,
           1003,  16011,  24051, 236761,    106,    107,    105,   4368,    107],
        [     2,    105,   9731,    107,   3048,    659,  11045,  16326, 236761,
            106,    107,    105,   2364,    107,   6974,    496,   2822,  31481,
           1003,  16011,  24051, 236761,    106,    107,    105,   4368,    107]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0'), 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0]], device='cuda:

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=1024
)

In [ ]:
response = processor.decode(outputs[:, input_len:], skip_special_tokens=False)

In [ ]:
response

['Why did the computer break up with the RAM?\n\nBecause it felt like their relationship was just hogging all the processing power!<turn|>',
 'Why did the programmer break up with the RAM chip?\n\nBecause he felt their relationship was taking up **too much memory**!<turn|>']

# Quantize

In [ ]:
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig, AutoProcessor, AutoModelForMultimodalLM
from trl import SFTConfig, SFTTrainer
import os

MODEL_NAME = "google/gemma-4-E4B-it"
processor = AutoProcessor.from_pretrained(MODEL_NAME)

W1005 14:13:35.681000 17092 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [ ]:
# The base model has *15882.207856* MB of size. Do not run this
# memory_model_b4_quant = model.get_memory_footprint() / 1e6
# print(f"Before Quantization Memory Model : {memory_model_b4_quant} MB")

Before Quantization Memory Model : 15882.207856 MB


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype= torch.float32,
)

model = AutoModelForMultimodalLM.from_pretrained(MODEL_NAME, quantization_config= bnb_config).to("cuda")

Loading weights: 100%|██████████| 2076/2076 [00:37<00:00, 55.44it/s] 


In [ ]:
memory_model_after_quant = model.get_memory_footprint() / 1e6
print(f"After Quantization Memory Model : {memory_model_after_quant} MB")
print(f"Difference Before and After Quantize The Model : {abs(memory_model_after_quant - 15882.207856)} MB")

After Quantization Memory Model : 9229.877872 MB
Difference Before and After Quantize The Model : 6652.329984000002 MB


## Example

In [ ]:
message1 = [
    {"role": "system", "content": "You are helpful assistant."},
    {"role": "user", "content": "Write a short joke about saving RAM."}
]

message2 = [
    {"role" : "system", "content" : "You are not helpful assistant at all."},
    {"role" : "user", "content" : "Write a joke about saving RAM."}
]

messages = [message1, message2]

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

In [ ]:
processor.tokenizer.convert_ids_to_tokens(0)

'<pad>'

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=1024
)

In [ ]:
response = processor.batch_decode(outputs[:, input_len:], skip_special_tokens=False)

In [ ]:
response

['<|channel>thought\nHere\'s a thinking process that leads to the suggested jokes:\n\n1.  **Analyze the Request:** The user wants a "short joke about saving RAM."\n2.  **Identify Key Concepts/Keywords:** RAM (Random Access Memory), computer, memory management, freeing up resources, speed, slowdown, allocation, byte, gigabyte.\n3.  **Determine the Tone:** It should be lighthearted, relatable to tech users, and quick.\n4.  **Brainstorm Angles (Metaphors/Puns):**\n    *   *Memory/Forgetfulness:* Relating RAM to human memory.\n    *   *Resource Management:* The struggle of allocating space.\n    *   *Performance:* The difference between having enough RAM and not having enough.\n    *   *The Act of Saving:* The effort involved in optimizing.\n\n5.  **Drafting Attempts (Self-Correction/Refinement):**\n\n    *   *(Attempt 1 - Too Technical):* Why did the computer sleep? Because it ran out of allocated virtual pages. (Too niche, not punchy.)\n    *   *(Attempt 2 - Focusing on the struggle):* I

In [ ]:
from IPython.display import display, Markdown

display(Markdown(response[0]))
display(Markdown(response[1]))

<|channel>thought
Here's a thinking process that leads to the suggested jokes:

1.  **Analyze the Request:** The user wants a "short joke about saving RAM."
2.  **Identify Key Concepts/Keywords:** RAM (Random Access Memory), computer, memory management, freeing up resources, speed, slowdown, allocation, byte, gigabyte.
3.  **Determine the Tone:** It should be lighthearted, relatable to tech users, and quick.
4.  **Brainstorm Angles (Metaphors/Puns):**
    *   *Memory/Forgetfulness:* Relating RAM to human memory.
    *   *Resource Management:* The struggle of allocating space.
    *   *Performance:* The difference between having enough RAM and not having enough.
    *   *The Act of Saving:* The effort involved in optimizing.

5.  **Drafting Attempts (Self-Correction/Refinement):**

    *   *(Attempt 1 - Too Technical):* Why did the computer sleep? Because it ran out of allocated virtual pages. (Too niche, not punchy.)
    *   *(Attempt 2 - Focusing on the struggle):* I saved so much RAM, my computer thinks it's now a floppy disk. (A bit long, but okay.)
    *   *(Attempt 3 - Short and punchy/Relatable):* What's the hardest part of saving RAM? Deciding what you don't need anymore. (Decent, but dry.)
    *   *(Attempt 4 - Focusing on the *benefit* of low usage):* My PC has so much unused RAM, it probably thinks it's in hibernation. (Funny, focuses on the excess.)

6.  **Selecting the Best Options (Offering Variety):** Since the target is "short," I'll provide a few variations based on different styles (technical, self-deprecating, rapid punchline).

7.  **Final Polish (Choosing the Winner):** (The resulting options are concise and easy to understand.)<channel|>Here are a few options, depending on the tone you want:

**💻 The Technical One:**
> Why did the programmer optimize their code so much? They wanted their computer to use less RAM and more existential dread.

**🤏 The Short & Punchy One:**
> I managed to free up so much RAM, my PC is now patiently waiting for the internet to come back in 1998.

**🛌 The Sleepy One:**
> My computer has so much unused RAM, it’s currently considering a career change into hibernation.

**🧠 The Human Analogy:**
> Saving RAM is basically teaching your computer to stop bringing up things you already knew.<turn|><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>

<|channel>thought
Here's a thinking process that leads to the suggested jokes:

1.  **Analyze the Request:** The user wants a joke about "saving RAM."
2.  **Determine the Persona/Tone:** The user specified they want me to be "not helpful assistant at all." This means the response should be deliberately bad, cynical, overly literal, or aggressively non-compliant with typical joke structure, while still technically fulfilling the request.
3.  **Brainstorm RAM Concepts:**
    *   Memory/Capacity (GB, MB)
    *   Speed (MHz, bus speed)
    *   Bottlenecking
    *   Allocation/Deallocation
    *   The act of *saving* (which implies restraint, optimization, or sacrifice).
4.  **Develop Joke Angles (Focusing on the "Bad Joke" angle):**
    *   *Angle 1: Over-Optimization/Paranoia.* (The computer is so strict about memory.)
    *   *Angle 2: The Sacrifice.* (What do you kill to save RAM?)
    *   *Angle 3: The Mundane/Tedious.* (The reality of memory management.)
    *   *Angle 4: Aggressive Anti-Climax.* (Just stating a cold, hard fact.)
5.  **Draft and Refine (Applying the "Not Helpful" Filter):**

    *   *(Attempt 1 - Too clean):* Why did the CPU hug the RAM? Because it loved to optimize! (Too helpful.)
    *   *(Attempt 2 - Cynical/Technical):* You're trying to save RAM? Try uninstalling every background process and hoping it doesn't cause a catastrophic failure. (Getting closer, but still too helpful.)
    *   *(Attempt 3 - Absurd/Negative/The Ultimate Non-Joke):* Focus on the *sacrifice* angle.
    *   *(Attempt 4 - Aggressive Micro-Management):* Make it sound like a bureaucratic nightmare.

6.  **Select the Best Options (Presenting a variety of bad jokes):** The final choices should embody digital misery and technical jargon misuse. (Self-Correction: Ensure the jokes are *about* saving RAM, not just general computer jokes.)

7.  **Final Output Generation.** (The chosen response delivers several options, each adhering to the deliberately abysmal standard requested.)<channel|>Why does a developer save RAM?

Because they fear the inevitable garbage collection, and they are equally terrified of the quarterly performance review.

***

**A longer one:**

Saving RAM is like trying to hold onto a receipt from a five-year-old transaction. It's a desperate, increasingly futile effort to prevent catastrophic overflow, fueled entirely by the paranoid delusion that everything is about to crash at 3 AM.

***

**The cold, harsh reality:**

You don't "save" RAM. You merely allocate it until the moment it realizes it has been utterly defeated by the next rendering task.

***

**(Bonus, if you need it to be aggressively unhelpful):**

Just run fewer things. The machine doesn't need your clutter. It needs *capacity*. And capacity is an illusion.<turn|>

# QLoRA

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
memory_model_after_quant = model.get_memory_footprint() / 1e6
print(f"After preparing model for kbit training. Memory Model : {memory_model_after_quant} MB")

After preparing model for kbit training. Memory Model : 16242.306224 MB


In [ ]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear4bit(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_fea

In [ ]:
config = LoraConfig(
    r=8, # the lower, the fewer parameters need to train
    lora_alpha=16, # usually 2*r
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=r".*language_model.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
)
model = get_peft_model(model, config)

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (vision_tower): Gemma4VisionModel(
          (patch_embedder): Gemma4VisionPatchEmbedder(
            (input_proj): Linear4bit(in_features=768, out_features=768, bias=False)
          )
          (encoder): Gemma4VisionEncoder(
            (rotary_emb): Gemma4VisionRotaryEmbedding()
            (layers): ModuleList(
              (0-15): 16 x Gemma4VisionEncoderLayer(
                (self_attn): Gemma4VisionAttention(
                  (q_proj): Gemma4ClippableLinear(
                    (linear): Linear4bit(in_features=768, out_features=768, bias=False)
                  )
                  (k_proj): Gemma4ClippableLinear(
                    (linear): Linear4bit(in_features=768, out_features=768, bias=False)
                  )
                  (v_proj): Gemma4ClippableLinear(
                    (linear): Linear4bit(in_features=768, out_features=768, bi

In [ ]:
memory_model_after_quant = model.get_memory_footprint() / 1e6
print(f"After preparing model for kbit training. Memory Model : {memory_model_after_quant} MB")

After preparing model for kbit training. Memory Model : 16312.069296 MB


In [ ]:
model.base_model.model.model.language_model

Gemma4TextModel(
  (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
  (layers): ModuleList(
    (0-4): 5 x Gemma4TextDecoderLayer(
      (self_attn): Gemma4TextAttention(
        (q_proj): lora.Linear4bit(
          (base_layer): Linear4bit(in_features=2560, out_features=2048, bias=False)
          (lora_dropout): ModuleDict(
            (default): Dropout(p=0.05, inplace=False)
          )
          (lora_A): ModuleDict(
            (default): Linear(in_features=2560, out_features=8, bias=False)
          )
          (lora_B): ModuleDict(
            (default): Linear(in_features=8, out_features=2048, bias=False)
          )
          (lora_embedding_A): ParameterDict()
          (lora_embedding_B): ParameterDict()
          (lora_magnitude_vector): ModuleDict()
        )
        (q_norm): Gemma4RMSNorm()
        (k_norm): Gemma4RMSNorm()
        (v_norm): Gemma4RMSNorm()
        (k_proj): lora.Linear4bit(
          (base_layer): Linear4bit(in_features=2560, 

In [ ]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Trainable parameters:      {train_p:,}')
print(f'Total parameters:          {tot_p:,}')
print(f'% of trainable parameters: {100*train_p/tot_p:.2f}%')

Trainable parameters:      17,440,768
Total parameters:          7,958,541,600
% of trainable parameters: 0.22%


# Training

## Dataset

In [ ]:
dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset

Dataset({
    features: ['sentence', 'translation', 'translation_extra'],
    num_rows: 720
})

In [ ]:
dataset[0]

{'sentence': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'translation_extra': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [ ]:
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role" : "user", "content" : examples["prompt"][i]},
                {"role" : "assistant", "content" : examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {"messages" : output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

In [ ]:
dataset = dataset.rename_columns({
    "sentence" : "prompt",
    "translation_extra" : "completion"
})
dataset = dataset.map(format_dataset)

In [ ]:
dataset = dataset.remove_columns(["prompt", "translation", "completion"])

In [ ]:
messages = dataset[0]["messages"]
messages

[{'role': 'user', 'content': 'The birch canoe slid on the smooth planks.'},
 {'role': 'assistant',
  'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}]

In [ ]:
processor.apply_chat_template(messages)

'<bos><|turn>user\nThe birch canoe slid on the smooth planks.<turn|>\n<|turn>model\nOn the smooth planks, the birch canoe slid. Yes, hrrrm.<turn|>\n'

## Training

In [ ]:
sft_config = SFTConfig(
    # Group 1: Memory Usage
    gradient_checkpointing=True, # saves memory
    gradient_checkpointing_kwargs={"use_reentrant" : True}, # avoid exception on newer versions of PyTorch
    gradient_accumulation_steps=2, # keep the loss until n batch
    
    per_device_train_batch_size=8, # batch size
    auto_find_batch_size=True, # prevent OOM and find the best batch size best on device performance

    # Group 2: Dataset-related
    max_length=128, # depends on max length on the dataset
    packing=False,
    packing_strategy="bfd",

    # Group 3: Training Parameters
    num_train_epochs=10,
    learning_rate=3e-4,
    optim="paged_adamw_8bit", # based on the article, 8-bit Adam Optimizer doesn't help much if using LoRA

    # Group 4: Logging parameters
    logging_steps=10,
    logging_strategy="epoch",
    output_dir="./qlora_gemma_4_e4b",
    report_to="none",

    bf16=True,    
)

In [ ]:
trainer = SFTTrainer(
    model = model.base_model.model,
    peft_config= config,
    processing_class=processor,
    args=sft_config,
    train_dataset=dataset
)

c:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\peft\mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating train dataset: 100%|██████████| 720/720 [00:00<00:00, 14722.09 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 720/720 [00:00<00:00, 28186.74 examples/s]


In [ ]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [ ]:
batch["input_ids"].shape

torch.Size([8, 42])

## Train Time

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss


KeyboardInterrupt: 